In [25]:
import pandas as pd
import glob
import re
import os
from functions import *

Clean NTN 

In [74]:
# Grab all CSV files
folder = "data_origin/omi_ntn/residential"

all_files = glob.glob(os.path.join(folder, "*.csv"))

# Concatenate all datasets into a single DataFrame
dfs = []

for i,f in enumerate(all_files):
    """
    Read every dataset in the folder as input.

    Extract year and semester from the file name.

    Select relevant columns.

    Return datasets with updated istat codes. 
    """
    # Extract filename without extension
    filename = os.path.splitext(os.path.basename(f))[0]

    # Extract year from filename
    match = re.search(r"_(\d{4})(?:_|$)", filename)
    year = int(match.group(1))
    
    # Read CSV
    df = pd.read_csv(f, sep=';')
    
    # Assign year to a new column
    df['year'] = year

    # Drop unnecessary columns
    df = df.drop(columns=['area', 'reg_name', 'prov_name'])

    # Strip commas from column names and NTN values
    df.columns = df.columns.str.strip(",")
    df["NTN"] = df["NTN"].astype(str).str.strip(",")

    # Use ',' as the decimal separator and convert to numeric (exlcude land_code)
    cols_to_convert = ["upto_50", "50_85", "85_115", "115_145", "over_145", "NTN"]
    df[cols_to_convert] = df[cols_to_convert].apply(
        lambda col: col.astype(str).str.replace(",", ".").astype(float)
    )    
    
    dfs.append(df)

# Concatenate all DataFrames into one
final_df = pd.concat(dfs, ignore_index=True)

# Sort by year, land_code
final_df = final_df.sort_values(by=['year', 'land_code'])

# Drop null values
final_df = final_df.dropna()

In [86]:
df_ntn = final_df.copy()

Import ISTAT codes

In [80]:
# Import ISTAT codes
df_istat = pd.read_csv('datasets/mun_istat_codes_non_normalized.csv')

In [87]:
# Merge with ISTAT codes
df_ntn = pd.merge(df_ntn, df_istat[['land_code', 'mun_istat', 'mun_name', 'prov_name', 'reg_name']], on = 'land_code', how = 'left')

# Drop land_code column
df_ntn = df_ntn.drop(columns = ['land_code'])

# Drop rows with missing values
df_ntn = df_ntn.dropna()

# Normalize ISTAT codes
df_ntn['mun_istat'] = df_ntn['mun_istat'].astype(int)
add_zeroes(df_ntn, ['mun_istat'], 6)

,upto_50,50_85,85_115,115_145,over_145,NTN,year,mun_istat,mun_name,prov_name,reg_name
0,5.17,18.49,48.16,38.67,32.18,142.67,2014,028001,Abano Terme,Padova,Veneto
1,0.00,0.00,4.00,0.00,0.00,4.00,2014,098001,Abbadia Cerreto,Lodi,Lombardia
2,7.00,7.50,8.00,3.11,0.00,25.61,2014,097001,Abbadia Lariana,Lecco,Lombardia
3,3.00,7.17,15.42,9.00,6.92,41.51,2014,052001,Abbadia San Salvatore,Siena,Toscana
4,2.00,3.00,3.00,3.00,3.17,14.17,2014,115001,Abbasanta,Oristano,Sardegna
...,...,...,...,...,...,...,...,...,...,...,...
83583,0.00,0.00,0.00,1.25,1.25,2.50,2024,005122,Moransengo-Tonengo,Asti,Piemonte
83584,5.00,20.00,22.38,22.00,6.17,75.55,2024,013256,Uggiate con Ronago,Como,Lombardia
83585,0.00,13.00,10.21,14.07,20.67,57.95,2024,024128,Sovizzo,Vicenza,Veneto
83586,6.11,18.86,15.50,17.33,22.93,80.73,2024,025075,Setteville,Belluno,Veneto


In [90]:
df_ntn = df_ntn.rename(columns = {
    "upto_50" : "Up to 50m²",
    "50_85" : "50-85m²",
    "85_115" : "85-115m²",
    "115_145" : "115-145m²",
    "over_145" : "Over 145m²",
    "NTN" : "Total transactions",
    "year" : "Year",
    "mun_name" : "Mun. name",
    "prov_name" : "Prov. name",
    "reg_name" : "Reg. name"
})

Save data

In [92]:
df_ntn.to_parquet('datasets/omi_ntn/omi_ntn_residential.parquet', index = False)